# Fine-tune Qwen3.5 in Unsloth Studio: support messages to ticket JSON

In this workshop you train a small model to turn a customer's support message into a ticket with a fixed set of fields. You do the training in Unsloth Studio, which is already running on this machine at **http://localhost:8890**, and use this notebook to look at the data and check the results.

```
"Case #99570. Customer Elena reports the rice cooker is not the        {"order_id": "ORD-10968", "customer": "Elena",
 model I ordered (ref ORD-10968). Order placed 24 January 2026.   ->    "phone": "(815) 781-9664", "product": "rice cooker",
 Total was USD 867.25. Call me on (815) 781-9664."                       "category": "wrong_item", "purchase_date": "2026-01-24",
                                                                         "amount": 867.25}
```

| Part | Where | Time |
|---|---|---|
| 1. Start training | Studio | 5 min |
| 2. Look at the data while it trains | this notebook | 10 min |
| 3. Check the run and compare answers | Studio | 10 min |
| 4. Score the model | this notebook | 10 min |
| 5. Export | Studio | 5 min |

Training takes about 3 to 5 minutes on this machine's Radeon GPU, so you start it first and read about the data while it runs.

Studio trains with supervised fine-tuning (SFT): you give it example inputs with the exact answers you want, and the model learns to produce answers like them. That suits this task because every message has one correct ticket.

## 1. Start training in Studio

Open **http://localhost:8890** in a new browser tab and sign in with the password `RyzenUnslothStudio`.

![Studio home](docs/studio-screenshots/01-studio-home.png)

### 1a. Pick the model

Go to **Train**. Under **Model**, open the list and choose `unsloth/Qwen3.5-2B`. It is already downloaded, so it appears under **On Device**. It is a 2-billion-parameter model, small enough to train in a few minutes.

![Model picker](docs/studio-screenshots/02-model-picker.png)

Set **Method** to **LoRA**. With LoRA the original model stays frozen and Studio trains a small set of extra weights next to it, here about 1% of the model's size.

![Training method](docs/studio-screenshots/03-training-method.png)

### 1b. Upload the data

Click **Drop file or click to upload**, paste `%USERPROFILE%\local-ai-workshop\unsloth\workshop\data\support_tickets_train.jsonl` into the file name box and press Enter. Then click **View dataset**. It should say *Detected ChatML conversation column*, which means Studio read each row as a conversation.

![Dataset preview](docs/studio-screenshots/04-dataset-preview.png)

### 1c. Start the run

Under **Parameters**, set the project name to `support-tickets` and **Max Steps** to **30**. Leave the other settings as they are, check the **Run preview** on the right, and click **Start Training**. Then come back here while it runs.

Each step trains on 8 examples, so 30 steps covers about a quarter of the 1,000 rows. That turned out to be enough: in our tests, 30 steps got 29 of 30 test tickets exactly right, and 60 steps got all 30 in twice the time.

![Run configured](docs/studio-screenshots/05-run-configured.png)

## 2. Look at the data while it trains

`data/support_tickets_train.jsonl` has the 1,000 messages you just uploaded, each with the ticket we want back. `data/support_tickets_eval.jsonl` has 100 more that the model never sees during training; we use them to score it in part 4.

In [1]:
import collections, json, pathlib, subprocess, sys

DATA = pathlib.Path("data")
if not (DATA / "support_tickets_train.jsonl").exists():
    subprocess.run([sys.executable, "scripts/generate_support_ticket_dataset.py"], check=True)

train = [json.loads(line) for line in open(DATA / "support_tickets_train.jsonl")]
evalset = [json.loads(line) for line in open(DATA / "support_tickets_eval.jsonl")]
print(f"{len(train)} training rows, {len(evalset)} held-out rows\n")

for row in train[:3]:
    user, ticket = row["messages"][0]["content"], row["messages"][1]["content"]
    print(user, "\n  ->", ticket, "\n")

print(collections.Counter(json.loads(r["messages"][1]["content"])["category"] for r in train))

1000 training rows, 100 held-out rows

Convert this support message to a ticket JSON.

Hi, my name is Omar. I'm writing about order ORD-49345: the office chair seems to have been lost by the courier. It cost USD 978.11. 
  -> {"order_id": "ORD-49345", "customer": "Omar", "phone": null, "product": "office chair", "category": "not_delivered", "purchase_date": null, "amount": 978.11} 

Convert this support message to a ticket JSON.

Hi, my name is Yuki. I'm writing about order ORD-79564: the wireless headphones stopped working after a week. It cost $1,052.37. Call me on (511) 915-3306. 
  -> {"order_id": "ORD-79564", "customer": "Yuki", "phone": "(511) 915-3306", "product": "wireless headphones", "category": "defective", "purchase_date": null, "amount": 1052.37} 

Convert this support message to a ticket JSON.

Hello support, the air purifier from order ORD-16560 arrived three weeks late. I paid $1,277.17. Call me on (907) 232-1671. Thanks, Raj 
  -> {"order_id": "ORD-16560", "customer": 

Things to notice in these examples:

- The instruction is a single line. It never lists the fields, the seven categories, the date format, or that missing values should be `null`. The model has to pick all of that up from the examples.
- Some fields need more than copying. "isn't what I expected" means `return_request`. Dates like `Jun 5, 2026` and `24 January 2026` all become `2026-06-05`-style dates, and `USD 1,007.56` becomes the number `1007.56`.
- Case numbers and tracking numbers look like IDs but never go in the ticket.
- Each row is a short conversation: the message from the user and the ticket as the assistant's reply. Studio formats it the same way its Chat page does, so what you see in Chat matches what was trained.

## 3. Check the run and compare answers

Go back to Studio's **Current Run** page. If the step counter is still at 0, Studio is loading the model and preparing the data. After that the counter moves and the charts fill in.

![Training in progress](docs/studio-screenshots/06-training-in-progress.png)

The **Training Loss** chart should drop steeply over the first 10 to 15 steps and then flatten out near zero. If it does not drop at all, Studio probably did not read the data as conversations; go back to step 1b.

![Training complete](docs/studio-screenshots/07-training-complete.png)

### Compare the two models in Chat

When the run finishes, click **Compare in Chat**. Studio opens the original model and your fine-tuned one side by side and sends each message to both.

Check that **Thinking** is off (the lightbulb next to the send button is grey). With Thinking on, the model reasons before answering, which is not how it was trained, and its answers drift from the ticket format.

Paste in a message the model has never seen. The next cell prints one.

In [2]:
# A held-out message to paste into Compare in Chat, and the ticket we hope to get back.
sample = evalset[11]
print(sample["messages"][0]["content"])
print("\nexpected:", sample["messages"][1]["content"])

Convert this support message to a ticket JSON.

Case #99570. Customer Elena reports the rice cooker is not the model I ordered (ref ORD-10968). Order placed 24 January 2026. Total was USD 867.25. Call me on (815) 781-9664.

expected: {"order_id": "ORD-10968", "customer": "Elena", "phone": "(815) 781-9664", "product": "rice cooker", "category": "wrong_item", "purchase_date": "2026-01-24", "amount": 867.25}


![Compare in Chat](docs/studio-screenshots/08-compare-in-chat.png)

The original model writes something ticket-like, but with its own field names and layout. The fine-tuned model answers with the fields we trained it on. To see whether that holds up beyond one message, score it on more of them.

## 4. Score the model

Here we run three setups on the same held-out tickets:

1. The original model with the same one-line instruction used in training.
2. The original model with a long prompt that spells out every field, the category list, the date format and the null rule. This is what you would do without fine-tuning.
3. The fine-tuned model with the one-line instruction.

Each answer is scored three ways: whether it is plain JSON with nothing around it, how many of the seven fields are correct, and whether the whole ticket is exactly right.

The next cell finds the model Studio just saved; it picks the newest `support-tickets` run. If it picks the wrong one, set `ADAPTER_DIR` to the path shown at the top of Studio's **Current Run** page after "Model saved to".

In [3]:
import glob, os

roots = [os.environ.get("UNSLOTH_STUDIO_HOME", ""), os.path.expanduser("~/.unsloth/studio")]
runs = sorted({p for r in roots if r for p in glob.glob(os.path.join(r, "outputs", "*project-support-tickets*"))},
              key=os.path.getmtime)
ADAPTER_DIR = runs[-1] if runs else None   # or the path from "Model saved to ..."
print(ADAPTER_DIR)

C:\Users\test\local-ai-workshop\unsloth\outputs\unsloth_Qwen3.5-2B__project-support-tickets_1791920197


In [4]:
# Load the base model with your adapter on top. The same model answers as "base"
# with the adapter switched off, so only one copy is in memory.
sys.path.insert(0, "scripts")
from score_tickets import SCHEMA_PROMPT, generate, score
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(ADAPTER_DIR, load_in_4bit = False)
FastModel.for_inference(model);

[bitsandbytes.cextension|WARNING]WARNING: BNB_ROCM_VERSION=715 environment variable detected; loading libbitsandbytes_rocm715.dll.
This overrides automatic ROCm version selection.
If this was unintended clear the variable and retry: unset BNB_ROCM_VERSION



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


C:\Users\test\local-ai-workshop\unsloth\unsloth_studio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1013 12:48:44.800000 6484 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


C:\Users\test\local-ai-workshop\unsloth\unsloth_studio\Lib\site-packages\unsloth\_gpu_init.py:456: UserWarning: Unsloth: torchcodec is installed but cannot load its FFmpeg libraries; install an FFmpeg full-shared build only if you need torchcodec itself; audio datasets decode through soundfile and PyAV instead (wav/flac/mp3/ogg, m4a/aac/webm).
  disable_torchcodec_if_broken()


...

Loading weights:  47%|████▋     | 293/617 [00:02<00:01, 252.83it/s]

Loading weights:  53%|█████▎    | 327/617 [00:02<00:01, 275.02it/s]

Loading weights:  71%|███████▏  | 440/617 [00:02<00:00, 513.88it/s]

Loading weights:  90%|████████▉ | 553/617 [00:02<00:00, 684.61it/s]

Loading weights: 100%|██████████| 617/617 [00:02<00:00, 247.94it/s]

In [5]:
# 30 held-out tickets per setup; raise N for a tighter estimate if you have time.
N = 30
rows = evalset[:N]
short = [r["messages"][0]["content"] for r in rows]
schema = [SCHEMA_PROMPT + "\n\n" + p.split("\n\n", 1)[1] for p in short]

answers = {}
with model.disable_adapter():
    answers["base, short prompt"] = generate(model, tokenizer, short)
    answers["base, schema prompt"] = generate(model, tokenizer, schema)
answers["fine-tuned, short prompt"] = generate(model, tokenizer, short)

import pandas as pd
pd.DataFrame({name: score(rows, a) for name, a in answers.items()}).T.style.format("{:.0%}")

,bare JSON,fields correct,whole ticket exact
"base, short prompt",0%,7%,0%
"base, schema prompt",100%,94%,63%
"fine-tuned, short prompt",100%,100%,97%


In [6]:
# The same held-out message through all three.
i = 3
print(short[i].split("\n\n", 1)[1], "\n")
for name, a in answers.items():
    print(f"--- {name}\n{a[i]}\n")

Hello support, the blender from order ORD-39968 won't turn on at all. I bought it on 07/11/2026. Thanks, Ben 

--- base, short prompt
```json
{
  "ticket_id": "TICKET-882910",
  "status": "open",
  "created_at": "2026-07-12T10:30:00Z",
  "updated_at": "2026-07-12T10:30:00Z",
  "subject": "Blender (ORD-39968) won't turn on",
  "customer": {
    "name": "Ben",
    "email": "ben@example.com"
  },
  "order": {
    "order_number": "ORD-39968",
    "purchase

--- base, schema prompt
{
  "order_id": "ORD-39968",
  "customer": "Ben",
  "phone": null,
  "product": "blender",
  "category": "not_delivered",
  "purchase_date": "2026-07-11",
  "amount": null
}


--- fine-tuned, short prompt
{"order_id": "ORD-39968", "customer": "Ben", "phone": null, "product": "blender", "category": "defective", "purchase_date": "2026-07-11", "amount": null}




Reading the results:

- With the one-line instruction, the original model has no idea which fields we want, so it scores close to zero.
- The long prompt gets the original model most of the way. Prompting alone is often good enough, and it is worth trying first. The tickets it gets wrong usually come down to details such as a category that needs judgment, or a guess where the answer should be `null`.
- The fine-tuned model gets the fields and rules from training and only needs the one-line instruction. On this machine it got 29 of 30 test tickets exactly right after 30 steps, against 19 of 30 for the long prompt. A shorter prompt is also cheaper to run.

If your fine-tuned model scores lower, look at the tickets it got wrong. Extra fields or text after the JSON usually mean it needs more steps. A wrong category usually means the training data has few examples phrased that way.

## 5. Export

In Studio, click **Export Model** on the Current Run page. There are three options:

- **LoRA Only** saves just the trained adapter, about 100 MB. You load it on top of the original model.
- **Merged Model** combines the two into one 16-bit model that Transformers or vLLM can load directly.
- **GGUF / Llama.cpp** makes a single compressed file for tools such as llama.cpp, LM Studio and Ollama. Studio can also convert a GGUF to Q4NX, the format for the NPU in AMD Ryzen AI processors.

![Export](docs/studio-screenshots/09-export.png)

In [7]:
# Free the GPU memory this notebook holds before training anything else in Studio.
del model
import gc, torch
gc.collect(); torch.cuda.empty_cache()

## Where to go next

- Try your own data. Studio accepts JSONL with a `messages` column like this dataset, or Alpaca-style `instruction` / `input` / `output` columns. A few hundred clean examples of the exact output you want is a good start.
- Try a larger model. `unsloth/Qwen3.5-9B` uses the same steps and handles harder tasks. Stay with Qwen models for chat-format data for now: Gemma models currently train slightly differently in Studio from how Studio serves them.
- When there is no single right answer but you can score an answer, look at the GRPO Sudoku notebook in this repo, which trains from scores instead of examples.